 Multimodal Invoice Voice Assistant



**Project flow:** Invoice Image → Vision Understanding → Structured Data → Voice Input → Whisper ASR → Grounded LLM Reasoning → Text Sanitization → Neural TTS → Spoken Answer.

### Concepts covered
- Multimodal Generative AI and Vision AI
- Vision prompting, visual grounding, guardrails, and structured extraction
- Vision Transformers (ViT) and cross-attention — explained simply
- Audio fundamentals: sample rate, bit depth, and channels
- Speech-to-Text / ASR with Whisper
- Domain vocabulary prompting and transcription metadata
- Audio intelligence and speech analytics
- Text-to-Speech, neural TTS, voice personas, and speed control
- Markdown/text sanitization before speech synthesis
- Multimodal voice-assistant architecture and context fusion
- Diffusion models and text-to-image generation — explained as the visual-generation part of the multimodal stack


## 1. Project Architecture

This project treats the invoice image, customer question, and model instructions as different inputs.
The system first understands the image, then listens to the user, reasons over trusted invoice data, and speaks the answer.
The important idea is **multimodal fusion**: different types of information are combined before the final response.

**Concept / Method used:** Multimodal AI, context fusion, voice-assistant architecture.

In [ ]:
pipeline = [
    "Invoice image",
    "Vision model extracts structured invoice JSON",
    "User audio is transcribed with Whisper",
    "LLM answers using only verified invoice data",
    "Response is cleaned for speech",
    "Neural TTS generates spoken audio",
]
for i, step in enumerate(pipeline, 1):
    print(f"{i}. {step}")


## 2. Install Dependencies

We install the OpenAI-compatible SDK for model calls and Pillow for image handling.
The same SDK is used for vision, speech-to-text, text reasoning, and text-to-speech.
Keeping the tools in one client makes the end-to-end workflow easier to understand.

**Concept / Method used:** OpenAI-compatible API client, multimodal inference.

In [ ]:
!pip install -q openai pillow

## 3. Secure API Configuration

The API key is never written directly into this notebook.
In Google Colab, store it in **Secrets** using the name `api_key`.
For local Jupyter, use the `NAVIGATE_API_KEY` environment variable instead.

**Concept / Method used:** Secure credential handling, environment variables, Colab Secrets.

In [ ]:
import os
from openai import OpenAI

try:
    from google.colab import userdata
    api_key = userdata.get("api_key") or userdata.get("NAVIGATE_API_KEY")
except Exception:
    api_key = os.getenv("NAVIGATE_API_KEY")

if not api_key:
    raise ValueError(
        "API key not found. Add `api_key` to Colab Secrets or set "
        "`NAVIGATE_API_KEY` as an environment variable."
    )

client = OpenAI(
    api_key=api_key,
    base_url="https://apidev.navigatelabsai.com/"
)

print("API client initialized securely.")


## 4. Multimodal Vision Concepts

A vision model can understand both text and visual information from an image.
Vision Transformers (ViT) split an image into small patches and process those patches as useful visual tokens.
Cross-attention helps the model connect words in a prompt with relevant visual information.

**Concept / Method used:** Vision AI, ViT, visual tokens, cross-attention.

In [ ]:
vision_concepts = {
    "Vision AI": "Understands information from images.",
    "ViT": "Represents image patches as tokens for transformer processing.",
    "Cross-attention": "Connects language information with visual information.",
    "Visual grounding": "Keeps the answer tied to evidence present in the image.",
}
for name, meaning in vision_concepts.items():
    print(f"{name}: {meaning}")


## 5. Prepare the Invoice Image

For a real project, the invoice can come from a camera, scanner, or uploaded file.
This notebook lets you upload an invoice image directly in Colab.
The image becomes the visual input for the invoice-understanding stage.

**Concept / Method used:** Image ingestion, document understanding.

In [ ]:
from google.colab import files
from PIL import Image
from IPython.display import display

uploaded = files.upload()

image_files = [
    name for name in uploaded
    if name.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
]

if not image_files:
    raise ValueError("Please upload an invoice image.")

invoice_path = image_files[0]
invoice_image = Image.open(invoice_path)

print("Invoice file:", invoice_path)
print("Image size:", invoice_image.size)
display(invoice_image)


## 6. Visual Prompting for Invoice Extraction

A good visual prompt tells the model exactly what information to extract and what not to guess.
We ask for a fixed JSON structure so the next stages can work with reliable fields.
The prompt also uses a grounding rule: values must come from the invoice image.

**Concept / Method used:** Visual prompting, structured extraction, grounding, guardrails.

In [ ]:
import base64
import json

with open(invoice_path, "rb") as f:
    invoice_b64 = base64.b64encode(f.read()).decode("utf-8")

vision_prompt = """
You are an invoice document understanding assistant.

Read the invoice image and extract ONLY information that is visibly present.
Return one valid JSON object using exactly this structure:

{
  "vendor": "",
  "invoice_number": "",
  "invoice_date": "",
  "due_date": "",
  "bill_to": "",
  "items": [
    {
      "description": "",
      "quantity": 0,
      "rate": "",
      "amount": ""
    }
  ],
  "subtotal": "",
  "tax": "",
  "total": "",
  "payment_instructions": ""
}

Rules:
1. Return only raw JSON.
2. Do not invent, estimate, or calculate missing values.
3. If a field is not visible, use an empty string.
4. Preserve monetary values as written on the invoice.
"""


## 7. Vision Model: Image-to-Text and Structured Data

The vision model receives the invoice image together with the visual prompt.
It converts the document into structured JSON that can be searched and reasoned over later.
This is the main **image-to-text / visual understanding** hands-on step.

**Concept / Method used:** Vision-language model, image-to-text, document understanding, JSON extraction.

In [ ]:
vision_response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": vision_prompt},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{invoice_b64}"
                    }
                }
            ]
        }
    ],
    temperature=0.0
)

invoice_json_text = vision_response.choices[0].message.content.strip()

try:
    invoice_data = json.loads(invoice_json_text)
except json.JSONDecodeError:
    raise ValueError("Vision model did not return valid JSON.")

print(json.dumps(invoice_data, indent=2))


## 8. Visual Grounding and Guardrails Check

Grounding means the assistant should use evidence from the invoice instead of making up facts.
We check that the extracted result is a JSON object and that important fields exist.
In production, stronger validation can compare fields with OCR or a second extraction pass.

**Concept / Method used:** Visual grounding, hallucination guardrails, schema validation.

In [ ]:
required_fields = [
    "vendor", "invoice_number", "invoice_date",
    "due_date", "items", "subtotal", "tax", "total"
]

missing = [field for field in required_fields if field not in invoice_data]

if missing:
    print("Validation warning - missing fields:", missing)
else:
    print("Invoice structure passed the basic validation check.")

print("Items extracted:", len(invoice_data.get("items", [])))


## 9. Audio Fundamentals

Digital audio is represented using samples taken many times per second.
Sample rate tells us how many samples are captured per second, while bit depth affects amplitude resolution.
Channels describe whether the recording is mono, stereo, or another multi-channel format.

**Concept / Method used:** Sample rate, bit depth, channels, digital audio fundamentals.

In [ ]:
audio_concepts = {
    "Sample rate": "Number of audio samples captured per second, measured in Hz.",
    "Bit depth": "Number of bits used to represent each sample's amplitude.",
    "Channels": "Number of independent audio channels, such as mono or stereo.",
}
for concept, explanation in audio_concepts.items():
    print(f"{concept}: {explanation}")


## 10. Upload Voice Question

The user can ask a question about the invoice using a recorded audio file.
A WAV, MP3, M4A, or another supported audio file can be uploaded.
This creates the voice-input part of our multimodal assistant.

**Concept / Method used:** Voice ingestion, audio input, human-computer interaction.

In [ ]:
audio_uploaded = files.upload()

audio_files = [
    name for name in audio_uploaded
    if name.lower().endswith((".wav", ".mp3", ".m4a", ".mpeg", ".mpga", ".webm"))
]

if not audio_files:
    raise ValueError("Please upload a supported audio recording.")

audio_filename = audio_files[0]
print("Voice input:", audio_filename)


## 11. Speech-to-Text with Whisper

Whisper converts the spoken question into text that the language model can understand.
The transcript becomes the dynamic user input for the reasoning stage.
This is the main Automatic Speech Recognition (ASR) step.

**Concept / Method used:** Speech-to-Text, ASR, Whisper.

In [ ]:
with open(audio_filename, "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
    )

user_question = transcription.text.strip()

print("Transcribed question:")
print(user_question)


## 12. Advanced ASR: Domain Vocabulary and Metadata

Technical and business conversations may contain names that are easy to transcribe incorrectly.
A vocabulary prompt can guide the recognizer toward expected terms.
Verbose output can also provide useful metadata such as detected language and duration.

**Concept / Method used:** ASR vocabulary prompting, domain jargon, verbose transcription metadata.

In [ ]:
with open(audio_filename, "rb") as audio_file:
    detailed_transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        language="en",
        prompt="invoice, invoice number, subtotal, tax, total, due date, payment",
        response_format="verbose_json"
    )

print("Detected language:", getattr(detailed_transcription, "language", "N/A"))
print("Duration:", getattr(detailed_transcription, "duration", "N/A"))
print("Transcript:", detailed_transcription.text)


## 13. Audio Intelligence and Speech Analytics

After transcription, the spoken signal becomes searchable text.
That allows the same system to detect keywords, questions, or compliance phrases.
This idea is useful for support calls, meeting analysis, and voice-based customer service.

**Concept / Method used:** Audio intelligence, transcript analysis, keyword spotting.

In [ ]:
def find_keywords(transcript, keywords):
    text = transcript.lower()
    return {keyword: (keyword.lower() in text) for keyword in keywords}

keywords_to_check = ["total", "due date", "payment", "tax"]
keyword_report = find_keywords(user_question, keywords_to_check)

for keyword, found in keyword_report.items():
    print(f"{keyword}: {'found' if found else 'not found'}")


## 14. Multimodal Context Fusion

Now we have two different inputs: structured visual information and spoken language.
The assistant combines them with fixed instructions before asking the language model to answer.
This is a practical example of multimodal fusion: image-derived context plus voice-derived context.

**Concept / Method used:** Multimodal fusion, context assembly, cross-modal reasoning.

In [ ]:
context_for_reasoning = {
    "invoice": invoice_data,
    "user_question": user_question,
    "rules": [
        "Use only the supplied invoice data.",
        "Do not invent missing invoice information.",
        "If information is unavailable, say so clearly.",
        "Keep the answer short and natural for voice output.",
    ],
}

print(json.dumps(context_for_reasoning, indent=2))


## 15. Grounded LLM Reasoning

The language model answers the user's spoken question using the extracted invoice JSON.
The prompt explicitly prevents the model from adding information that is not in the invoice.
The result is kept short because it will be converted into speech.

**Concept / Method used:** LLM reasoning, grounding, prompt engineering, zero-hallucination guardrails.

In [ ]:
reasoning_prompt = f"""
You are a helpful invoice voice assistant.

Answer the user's question using ONLY the verified invoice JSON below.

VERIFIED INVOICE DATA:
{json.dumps(invoice_data, indent=2)}

USER QUESTION:
{user_question}

Rules:
- Do not invent or estimate information.
- Do not use outside knowledge to fill missing invoice fields.
- If the requested information is not available, say that it is not available.
- Keep the answer to 1 or 2 conversational sentences.
- Do not use markdown formatting.
- Write monetary amounts clearly for speech.
"""

reasoning_response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[{"role": "user", "content": reasoning_prompt}],
    temperature=0.0
)

assistant_text = reasoning_response.choices[0].message.content.strip()

print("Assistant text:")
print(assistant_text)


## 16. TTS Text Sanitization

Language models sometimes return markdown symbols, links, or code formatting.
Those symbols can sound unnatural when sent directly to a speech synthesizer.
We clean the text first so the spoken response sounds more natural.

**Concept / Method used:** Text preprocessing, TTS sanitization, speech-ready formatting.

In [ ]:
import re

def sanitize_text_for_tts(raw_text):
    text = re.sub(r"```.*?```", "", raw_text, flags=re.DOTALL)
    text = re.sub(r"`.*?`", "", text)
    text = re.sub(r"\[(.*?)\]\(.*?\)", r"\1", text)
    text = re.sub(r"#+\s*", "", text)
    text = re.sub(r"\*+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

speech_text = sanitize_text_for_tts(assistant_text)

print("Speech-ready text:")
print(speech_text)


## 17. Neural Text-to-Speech

The cleaned answer is converted from text into natural speech.
Neural TTS models learn patterns of pronunciation, rhythm, and voice generation.
The audio is then played directly inside the Colab notebook.

**Concept / Method used:** Text-to-Speech, neural TTS, speech synthesis.

In [ ]:
from IPython.display import Audio, display

tts_response = client.audio.speech.create(
    model="gemini-2.5-flash-tts",
    voice="alloy",
    input=speech_text
)

display(Audio(tts_response.content, autoplay=False))


## 18. Voice Persona and Speech Rate

Different voices can make the same assistant feel different to the listener.
Speech speed can also be adjusted for short alerts or slower explanations.
We test more than one voice setting to understand voice tuning.

**Concept / Method used:** Voice personas, speech-rate control, TTS tuning.

In [ ]:
voice_samples = [
    ("nova", 1.0),
    ("onyx", 0.9),
]

for voice_name, speed_rate in voice_samples:
    print(f"Voice: {voice_name} | Speed: {speed_rate}x")
    sample = client.audio.speech.create(
        model="gemini-2.5-flash-tts",
        voice=voice_name,
        speed=speed_rate,
        input=speech_text
    )
    display(Audio(sample.content, autoplay=False))


## 19. Diffusion Models and Text-to-Image Concepts

Diffusion models generate images by learning how to remove noise step by step.
A text prompt guides the generation process toward a desired visual result.
This is another multimodal capability: text can become a visual asset that can later be given to a vision model.

**Concept / Method used:** Diffusion models, text-to-image generation, prompt conditioning.

In [ ]:
generation_concepts = {
    "Diffusion": "Starts from noisy data and progressively denoises toward an image.",
    "Text conditioning": "The prompt guides the visual generation process.",
    "Prompt parameters": "Words, style, composition, and constraints influence the generated image.",
    "Multimodal loop": "Generated images can become visual input for a vision model.",
}

for concept, meaning in generation_concepts.items():
    print(f"{concept}: {meaning}")

print("\nNote: This capstone uses an uploaded invoice image as the actual visual input.")
print("The supplied hands-on material does not define a specific image-generation endpoint,")
print("so the text-to-image section explains the concept without assuming an unsupported API call.")


## 20. Build the Complete Voice Assistant Function

All major steps can now be wrapped into one reusable function.
The function receives an audio question, uses cached invoice data, generates a grounded answer, and speaks it.
This makes the notebook behave like a small end-to-end customer support assistant.

**Concept / Method used:** End-to-end multimodal pipeline, reusable function, cached context.

In [ ]:
def ask_invoice_voice(audio_path, voice="nova", speed=1.0):
    # 1. Speech-to-text
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file
        )

    question = transcript.text.strip()

    # 2. Grounded reasoning over cached invoice data
    prompt = f"""
You are an invoice voice assistant.

Use ONLY this verified invoice data:
{json.dumps(invoice_data, indent=2)}

User question:
{question}

Rules:
- Do not invent information.
- If information is unavailable, say so.
- Answer in 1 or 2 natural sentences.
- Do not use markdown.
"""

    answer_response = client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )

    answer = answer_response.choices[0].message.content.strip()
    answer = sanitize_text_for_tts(answer)

    # 3. Text-to-speech
    audio_response = client.audio.speech.create(
        model="gemini-2.5-flash-tts",
        voice=voice,
        speed=speed,
        input=answer
    )

    print("User:", question)
    print("Assistant:", answer)
    display(Audio(audio_response.content, autoplay=False))

    return {"question": question, "answer": answer}


## 21. Test the Complete Pipeline

This final test runs the whole multimodal loop from voice input to spoken response.
The invoice data is reused instead of extracting the image again, which reduces unnecessary model calls.
You can upload a new audio question and run the function again.

**Concept / Method used:** End-to-end testing, context caching, multimodal voice interaction.

In [ ]:
result = ask_invoice_voice(
    audio_filename,
    voice="nova",
    speed=1.0
)

print("\nStructured result:")
print(json.dumps(result, indent=2))


## 22. Optional Compliance / Keyword Scanner

The same Whisper transcript can be checked against required phrases.
This connects speech recognition with audio intelligence and simple compliance analysis.
The result clearly shows which required phrases are present or missing.

**Concept / Method used:** Speech analytics, keyword detection, compliance checking.

In [ ]:
def audit_recording(audio_path, required_phrases):
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file
        ).text.strip()

    lower_text = transcript.lower()
    missing = [
        phrase for phrase in required_phrases
        if phrase.lower() not in lower_text
    ]

    return {
        "status": "PASS" if not missing else "FAIL",
        "missing_phrases": missing,
        "transcript": transcript,
    }

# Example:
# report = audit_recording(audio_filename, ["payment", "invoice"])
# print(json.dumps(report, indent=2))


## 23. Final Concept Map

The project connects the lessons into one understandable pipeline.
Vision AI reads the document, ASR understands the user's voice, and the LLM reasons over grounded context.
TTS then turns the final answer back into speech, completing the voice interaction.

**Concept / Method used:** Full Day 5 integration.

In [ ]:
concept_map = [
    ("Vision", "Invoice image -> structured JSON"),
    ("ViT / Cross-attention", "Visual features connected with language"),
    ("Visual prompting", "Tell the model exactly what to extract"),
    ("Grounding", "Use only evidence from the invoice"),
    ("Audio fundamentals", "Sample rate, bit depth, channels"),
    ("ASR / Whisper", "Voice -> text"),
    ("Audio intelligence", "Search and analyze the transcript"),
    ("Context fusion", "Invoice JSON + user question + rules"),
    ("LLM reasoning", "Generate a grounded answer"),
    ("TTS", "Text -> spoken audio"),
    ("Voice tuning", "Persona + speech rate"),
    ("Diffusion", "Text -> image concept for multimodal generation"),
]

for stage, purpose in concept_map:
    print(f"{stage}: {purpose}")


## 24. Project Summary

You have built one complete multimodal voice assistant instead of separate small demos.
The project combines document vision, speech recognition, grounded reasoning, and speech synthesis.
The strongest engineering idea is to keep each context type clear and pass only the information needed at each stage.

**Final project:** Multimodal Invoice Voice Assistant — Image → Voice → Reasoning → Voice.

In [ ]:
print("Day 5 Capstone completed.")
print("Pipeline: Invoice Image -> Vision -> JSON -> Whisper -> Grounded LLM -> TTS")
print("Security: API key is loaded from Colab Secrets or an environment variable.")
